In [4]:
#!/usr/bin/env python3
"""
analyze_object_to_ttp_links_with_examples.py
--------------------------------------------

Analyzes object_to_ttp_links_filled.json to extract:

- Distribution of source types
- Distribution of relationship types
- Count of techniques (TTPs)
- Count of tactics (IDs starting with "TA")
- Example record per source_type
- Tactic counts per relationship_type

Outputs:
- Pretty printed statistics
- summary_stats.json
"""

import json
import os
from collections import Counter, defaultdict
import random

INPUT_FILE = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/mitre_relationships.json"
OUT_FILE = "summary_stats.json"


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------

def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def is_tactic(tid: str) -> bool:
    """Return True if the technique ID is a MITRE ATT&CK tactic (TAxxxx)."""
    return isinstance(tid, str) and tid.startswith("TA")


# ----------------------------------------------------------------------
# Core analysis
# ----------------------------------------------------------------------

def analyze_distribution(data):
    source_counter = Counter()
    reltype_counter = Counter()
    technique_counter = Counter()
    tactic_counter = Counter()

    # Per-source-type unique TTP lists
    source_to_ttps = defaultdict(set)

    # Per-relationship-type stats
    reltype_to_ttps = defaultdict(set)
    reltype_to_tactics = defaultdict(set)

    for record in data:
        src_type = record.get("source_type", "unknown")
        rel_type = record.get("relationship_type", "unknown")
        ttp_id = record.get("technique_id")

        # Count overall distributions
        source_counter[src_type] += 1
        reltype_counter[rel_type] += 1

        # Count techniques/tactics
        if ttp_id:
            technique_counter[ttp_id] += 1
            source_to_ttps[src_type].add(ttp_id)

            reltype_to_ttps[rel_type].add(ttp_id)

            # tactic detection
            if is_tactic(ttp_id):
                tactic_counter[ttp_id] += 1
                reltype_to_tactics[rel_type].add(ttp_id)

    summary = {
        "total_records": len(data),
        "total_unique_ttps": len(technique_counter),
        "total_unique_tactics": len(tactic_counter),
        "source_type_distribution": source_counter.most_common(),
        "relationship_type_distribution": reltype_counter.most_common(),
        "source_type_unique_ttps": {k: len(v) for k, v in source_to_ttps.items()},
        "relationship_type_unique_ttps": {k: len(v) for k, v in reltype_to_ttps.items()},
        "relationship_type_unique_tactics": {k: len(v) for k, v in reltype_to_tactics.items()},
        "tactics": tactic_counter.most_common(),
    }

    return summary, source_to_ttps, technique_counter


# ----------------------------------------------------------------------
# Generate example record per source type
# ----------------------------------------------------------------------

def get_examples_by_source_type(data):
    by_type = defaultdict(list)
    for rec in data:
        stype = rec.get("source_type", "unknown")
        by_type[stype].append(rec)

    examples = {stype: random.choice(records) for stype, records in by_type.items()}
    return examples


# ----------------------------------------------------------------------
# Printing functions
# ----------------------------------------------------------------------

def print_summary(summary):
    print("\n=== Summary Statistics ===")
    print(f"Total relationships: {summary['total_records']:,}")
    print(f"Unique techniques (TTPs): {summary['total_unique_ttps']:,}")
    print(f"Unique tactics (TAxxxx): {summary['total_unique_tactics']:,}\n")

    print("🔹 Source Type Distribution:")
    for stype, count in summary["source_type_distribution"]:
        print(f"  {stype:25s} {count:8,d} links")

    print("\n🔹 Unique TTPs per Source Type:")
    for stype, n_ttps in summary["source_type_unique_ttps"].items():
        print(f"  {stype:25s} {n_ttps:8,d} unique TTPs")

    print("\n🔹 Relationship Type Distribution:")
    for rtype, count in summary["relationship_type_distribution"]:
        print(f"  {rtype:25s} {count:8,d} links")

    print("\n🔹 Unique TTPs per Relationship Type:")
    for rtype, n_ttps in summary["relationship_type_unique_ttps"].items():
        print(f"  {rtype:25s} {n_ttps:8,d} unique TTPs")

    print("\n🔹 Unique Tactics (TAxxxx) per Relationship Type:")
    for rtype, n_tac in summary["relationship_type_unique_tactics"].items():
        print(f"  {rtype:25s} {n_tac:8,d} unique tactics")

    print("\n🔹 All tactic IDs (sorted):")
    for tid, count in summary["tactics"]:
        print(f"  {tid:25s} {count:8,d} references")


def top_techniques(technique_counter, top_n=15):
    print(f"\n=== Top {top_n} Most Referenced Techniques ===")
    for tid, count in technique_counter.most_common(top_n):
        print(f"  {tid:10s} {count:6d} references")


def print_examples(examples):
    print("\n=== Example Record per Source Type ===")
    for stype, rec in examples.items():
        desc = rec.get("relationship_description")
        if desc is None:
            desc = "(no description available)"
        elif len(desc) > 300:
            desc = desc[:300] + "..."

        print(f"\n📘 Source Type: {stype}")
        print(f"  Source Name:         {rec.get('source_name')}")
        print(f"  Relationship Type:   {rec.get('relationship_type')}")
        print(f"  Description:         {desc}")
        print(f"  Technique:           {rec.get('technique_id')} — {rec.get('technique_name')}")
        print(f"  Technique URL:       {rec.get('technique_url')}")
        print(f"  Direction:           {rec.get('direction')}")


# ----------------------------------------------------------------------
# Save summary to JSON
# ----------------------------------------------------------------------

def save_summary(summary, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"\n✅ Summary saved to {path}")


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ ERROR: {INPUT_FILE} not found.")
        return

    print(f"[+] Loading {INPUT_FILE} ...")
    data = load_data(INPUT_FILE)
    print(f"[+] Loaded {len(data)} relationship records")

    summary, source_to_ttps, technique_counter = analyze_distribution(data)
    examples = get_examples_by_source_type(data)

    print_summary(summary)
    top_techniques(technique_counter)
    print_examples(examples)

    save_summary(summary, OUT_FILE)


if __name__ == "__main__":
    main()


[+] Loading /home/simonettos/thijs/data_augmentatio_stefano/mitre/mitre_relationships.json ...
[+] Loaded 19277 relationship records

=== Summary Statistics ===
Total relationships: 19,277
Unique techniques (TTPs): 853
Unique tactics (TAxxxx): 14

🔹 Source Type Distribution:
  malware                      9,836 links
  intrusion-set                4,362 links
  course-of-action             1,445 links
  campaign                     1,019 links
  attack-pattern                 837 links
  tool                           800 links
  x-mitre-detection-strategy      691 links
  capec                          273 links
  mitre-attack                    14 links

🔹 Unique TTPs per Source Type:
  attack-pattern                 837 unique TTPs
  malware                        418 unique TTPs
  course-of-action               582 unique TTPs
  intrusion-set                  488 unique TTPs
  campaign                       297 unique TTPs
  tool                           251 unique TTPs
  x-mitre-

In [7]:
with open('datasets/enterprise-attack.json', 'r') as f:
    data = json.load(f )

#get an overview of data
print(len(data['objects']))
print(data['objects'][2])
types = sorted({obj["type"] for obj in data["objects"]})
print(len(types))
print(types)

# for t in types:
#     print(t, sum(1 for o in data["objects"] if o["type"] == t))
ext_ids=[]
for item in data["objects"]:
    if item["type"] == "attack-pattern":
        if (item['external_references'][0]['external_id']):
            ext_ids.append(item['external_references'][0]['external_id'])
            



19981
{'x_mitre_domains': ['enterprise-attack'], 'object_marking_refs': ['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'], 'id': 'course-of-action--02f0f92a-0a51-4c94-9bda-6437b9a93f22', 'type': 'course-of-action', 'created': '2018-10-17T00:14:20.652Z', 'created_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', 'external_references': [{'source_name': 'mitre-attack', 'url': 'https://attack.mitre.org/mitigations/T1151', 'external_id': 'T1151'}], 'modified': '2019-07-25T11:46:32.010Z', 'name': 'Space after Filename Mitigation', 'description': 'Prevent files from having a trailing space after the extension.', 'x_mitre_deprecated': True, 'x_mitre_version': '1.0', 'x_mitre_modified_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5'}
13
['attack-pattern', 'campaign', 'course-of-action', 'identity', 'intrusion-set', 'malware', 'marking-definition', 'relationship', 'tool', 'x-mitre-data-component', 'x-mitre-data-source', 'x-mitre-matrix', 'x-mitre-tactic']


In [8]:
with open ('/home/simonettos/thijs/data_augmentatio_stefano/mitre/mitre_relationships.json', 'r') as f:
    data_rel = json.load(f )

my_ext_ids=[]
for item in data_rel:
    if item["source_type"] == "attack-pattern":
        my_ext_ids.append(item["technique_id"])

print(len(ext_ids))
print(len(my_ext_ids))
print(set(my_ext_ids) - set(ext_ids))

780
837
{'T1569.003', 'T1672', 'T1668', 'T1059.013', 'T1669', 'T1027.015', 'T1680', 'T1213.004', 'T1204.005', 'T1677', 'T1629', 'T1219.003', 'T1480.002', 'T1036.010', 'T1027.017', 'T1675', 'T1564.013', 'T1671', 'T1219.002', 'T1564.014', 'T1496.003', 'T1673', 'T1496.001', 'T1213.006', 'T1127.002', 'T1562.013', 'T1176.002', 'T1036.012', 'T1098.007', 'T1059.011', 'T1496.004', 'T1027.016', 'T1219.001', 'T1213.005', 'T1204.004', 'T1027.014', 'T1176.001', 'T1036.011', 'T1558.005', 'T1496.002', 'T1070.010', 'T1546.018', 'T1667', 'T1059.012', 'T1071.005', 'T1437', 'T1666', 'T1546.017', 'T1679', 'T1674', 'T1678', 'T1557.004', 'T1505.006', 'T1518.002', 'T1485.001', 'T1127.003', 'T1681'}
